# LASSA SEROPREVALENCE MACHINE LEARNING
## SECTION 6: CROSS-REACTIVITY vs TRUE PCR STATUS MODEL (END-TO-END)

**Goal:**
- Build the best possible model (with current data) to estimate:
  1. Probability of **PCR positive** (`p_pcr_pos`)
  2. Probability of **high cross-reactivity but PCR negative** (`p_cross_reactive`)

    - High cross-reactivity ≈ high IgM/IgG OD but negative PCR (0)

We will:
- Load & clean data (same as Sections 3–5)
- Explicitly **select 1 positive as test**, use the other **2 positives + all negatives** for training
- Build a strong preprocessing + XGBoost pipeline
- Run **careful hyperparameter tuning** (within reason for 3 positives)
- Use **bagging (multiple random seeds)** to stabilize predictions
- Calibrate probabilities with **Platt scaling**
- Define logic to produce:
  - `p_pcr_pos` (calibrated)
  - `p_cross_reactive` = P(high OD | PCR negative)
- Save all artifacts for later deployment

⚠️ **CRITICAL WARNING**:
- Only **3 PCR positives** → all performance estimates are unstable.
- This is **research / exploration only**, not a final clinical tool yet.


---
## CELL 6.0: IMPORTS + SETUP


In [1]:
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
    confusion_matrix,
)
from sklearn.utils import shuffle

import xgboost as xgb
import joblib

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 7),
    "figure.dpi": 110,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Path("models").mkdir(parents=True, exist_ok=True)
Path("results/plots").mkdir(parents=True, exist_ok=True)
Path("results/reports").mkdir(parents=True, exist_ok=True)

print("\n" + "="*100)
print("SECTION 6: CROSS-REACTIVITY vs TRUE PCR STATUS MODEL")
print("="*100)
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Run timestamp:", RUN_TS)



SECTION 6: CROSS-REACTIVITY vs TRUE PCR STATUS MODEL
Run timestamp: 20260401_130512


---
## CELL 6.1: LOAD DATA + CLEAN + TARGET MAPPING

- Drop `Full_Name`, `Patient_ID`
- Drop high-cardinality `Town/City`, `Occupation`


In [3]:
# IMPORTANT: set this to your actual file
DATA_PATH = "data/embeddings/data/LASV_Master_Data!.csv"  # adjust if needed
TARGET_COL = "lab_results.PCR_Results"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Missing dataset at: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

# strip whitespace
for c in df.select_dtypes(include=["object"]).columns:
    df[c] = df[c].astype(str).str.strip()

# drop ID/leakage
df = df.drop(columns=["Full_Name", "Patient_ID"], errors="ignore")

# map target
mapping = {"No Kb (Negative)": 0, "320Kb (Positive)": 1}
y = df[TARGET_COL].map(mapping)
if y.isna().any():
    bad = df.loc[y.isna(), TARGET_COL].unique().tolist()
    raise ValueError(f"Unmapped target values found: {bad}")
y = y.astype(int).values
X = df.drop(columns=[TARGET_COL])

# drop high-cardinality
DROP_COLS = [c for c in ["Town/City", "Occupation"] if c in X.columns]
X = X.drop(columns=DROP_COLS, errors="ignore").copy()

print("Shape:", X.shape)
print("Dropped:", DROP_COLS)
print("Positives:", int((y==1).sum()), "Negatives:", int((y==0).sum()))


Shape: (250, 41)
Dropped: ['Town/City', 'Occupation']
Positives: 3 Negatives: 247


---
## CELL 6.2: MANUALLY SELECT 1 POSITIVE AS TEST, 2 POSITIVES AS TRAIN

- With only 3 positives, we **force** 1 positive into test set.
- All negatives are split normally (80/20) around that.
- This gives us: train (2 pos + ~80% neg), test (1 pos + ~20% neg).


In [9]:
from sklearn.model_selection import train_test_split

pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

if len(pos_idx) != 3:
    raise ValueError(f"Expected exactly 3 positives, found {len(pos_idx)}")

# Choose 1 positive as test, 2 positives as train
test_pos_idx = np.array([pos_idx[0]])
train_pos_idx = np.array(pos_idx[1:])

# Split negatives 80/20
X_neg = X.iloc[neg_idx]
y_neg = y[neg_idx]
X_neg_train, X_neg_test, y_neg_train, y_neg_test = train_test_split(
    X_neg, y_neg, test_size=0.2, random_state=RANDOM_SEED, stratify=y_neg
)

# Build final train/test indices
train_idx = np.concatenate([
    train_pos_idx,
    X_neg_train.index.values
])

test_idx = np.concatenate([
    test_pos_idx,
    X_neg_test.index.values
])

X_train = X.loc[train_idx].reset_index(drop=True)
y_train = y[train_idx]

X_test = X.loc[test_idx].reset_index(drop=True)
y_test = y[test_idx]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train positives:", int((y_train==1).sum()), "Train negatives:", int((y_train==0).sum()))
print("Test positives:", int((y_test==1).sum()), "Test negatives:", int((y_test==0).sum()))


Train shape: (199, 41) Test shape: (51, 41)
Train positives: 2 Train negatives: 197
Test positives: 1 Test negatives: 50


---
## CELL 6.3: PREPROCESSING PIPELINE

- Numeric: median impute + StandardScaler
- Categorical: most_frequent impute + OneHotEncoder


In [10]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Numeric cols:", len(num_cols), "Categorical cols:", len(cat_cols))


Numeric cols: 3 Categorical cols: 38


---
## CELL 6.4: XGBoost Model Factory + Hyperparameter Grid

We use a **small but meaningful grid** to avoid crazy overfitting with 2 positives.


In [11]:
def make_xgb(params):
    return xgb.XGBClassifier(
        n_estimators=params.get("n_estimators", 2000),
        learning_rate=params.get("learning_rate", 0.02),
        max_depth=params.get("max_depth", 3),
        subsample=params.get("subsample", 0.9),
        colsample_bytree=params.get("colsample_bytree", 0.9),
        reg_lambda=params.get("reg_lambda", 2.0),
        min_child_weight=params.get("min_child_weight", 2),
        gamma=params.get("gamma", 0),
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=params.get("scale_pos_weight", 1.0),
        random_state=params.get("random_state", RANDOM_SEED),
        n_jobs=-1
    )

neg_train = int((y_train == 0).sum())
pos_train = int((y_train == 1).sum())
spw_default = neg_train / max(pos_train, 1)
print("Train scale_pos_weight default:", spw_default)

param_grid = {
    "n_estimators": [1200, 2000, 2600],
    "learning_rate": [0.01, 0.02, 0.03],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "reg_lambda": [1.0, 2.0, 4.0],
    "min_child_weight": [1, 2, 4],
    "gamma": [0, 0.5, 1.0],
    "scale_pos_weight": [spw_default * 0.5, spw_default, spw_default * 1.5]
}


Train scale_pos_weight default: 98.5


---
## CELL 6.5: Nested CV for Hyperparameter Search (on TRAIN ONLY)

We use **StratifiedKFold with 2 folds** on train (because there are only 2 positives).

Metric focus:
- PR-AUC (primary)
- ROC-AUC (secondary)
- F1 for positive class (for ranking ties)


In [18]:
# WARNING: this grid can still be large. To reduce time, we sample a subset.
all_params = list(ParameterGrid(param_grid))
print("Total param combos:", len(all_params))

# To avoid huge compute, sample e.g. 40 random combos
np.random.seed(RANDOM_SEED)
if len(all_params) > 40:
    sampled_idx = np.random.choice(len(all_params), size=40, replace=False)
    search_params = [all_params[i] for i in sampled_idx]
else:
    search_params = all_params

print("Using param combos:", len(search_params))

skf2 = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_SEED)

results = []

Xt_full_train = preprocess.fit_transform(X_train)

for idx, p in enumerate(search_params, 1):
    fold_pr_aucs = []
    fold_roc_aucs = []
    fold_f1s = []

    for tr_idx, va_idx in skf2.split(Xt_full_train, y_train):
        Xt_tr, Xt_va = Xt_full_train[tr_idx], Xt_full_train[va_idx]
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]

        model = make_xgb({**p, "random_state": RANDOM_SEED})
        model.fit(Xt_tr, y_tr, eval_set=[(Xt_va, y_va)], verbose=False)

        proba_va = model.predict_proba(Xt_va)[:, 1]
        if len(np.unique(y_va)) > 1:
            pr_auc = average_precision_score(y_va, proba_va)
            roc_auc = roc_auc_score(y_va, proba_va)
        else:
            pr_auc = np.nan
            roc_auc = np.nan

        y_hat_va = (proba_va >= 0.5).astype(int)
        f1 = f1_score(y_va, y_hat_va, zero_division=0)

        fold_pr_aucs.append(pr_auc)
        fold_roc_aucs.append(roc_auc)
        fold_f1s.append(f1)

    results.append({
        "params": p,
        "pr_auc_mean": np.nanmean(fold_pr_aucs),
        "roc_auc_mean": np.nanmean(fold_roc_aucs),
        "f1_mean": np.nanmean(fold_f1s)
    })

results_df = pd.DataFrame(results).sort_values(
    ["pr_auc_mean", "roc_auc_mean", "f1_mean"], ascending=False
).reset_index(drop=True)

print("Top 5 configs:")
display(results_df.head(5))

best_params = results_df.iloc[0]["params"]
print("\nBest params:")
print(best_params)

# Save search results
results_df.to_csv("results/reports/section6_hyperparam_search.csv", index=False)


Total param combos: 19683
Using param combos: 40
Top 5 configs:


,params,pr_auc_mean,roc_auc_mean,f1_mean
0,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.063406,0.865105,0.0
1,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.048055,0.829803,0.0
2,"{'colsample_bytree': 0.8, 'gamma': 1.0, 'learn...",0.046316,0.811946,0.0
3,"{'colsample_bytree': 1.0, 'gamma': 1.0, 'learn...",0.026739,0.647985,0.0
4,"{'colsample_bytree': 1.0, 'gamma': 0.5, 'learn...",0.026739,0.647985,0.0



Best params:
{'colsample_bytree': 0.9, 'gamma': 0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 1200, 'reg_lambda': 1.0, 'scale_pos_weight': 49.25, 'subsample': 0.8}


---
## CELL 6.6: Train Final Ensemble on TRAIN (Bagging over Seeds)

We train several XGB models with different seeds using the **best hyperparameters**, then average their predicted probabilities.


In [19]:
Xt_train = preprocess.fit_transform(X_train)
Xt_test = preprocess.transform(X_test)

seeds = [11, 22, 33, 44, 55]
ensemble_models = []

for s in seeds:
    p = {**best_params, "random_state": s}
    model = make_xgb(p)
    model.fit(Xt_train, y_train, verbose=False)
    ensemble_models.append(model)

def ensemble_predict_proba(models, X):
    probs = [m.predict_proba(X)[:,1] for m in models]
    return np.mean(probs, axis=0)

train_proba_raw = ensemble_predict_proba(ensemble_models, Xt_train)
test_proba_raw = ensemble_predict_proba(ensemble_models, Xt_test)

print("Train raw PR-AUC:", average_precision_score(y_train, train_proba_raw))
print("Train raw ROC-AUC:", roc_auc_score(y_train, train_proba_raw))
print("Test raw PR-AUC:", average_precision_score(y_test, test_proba_raw))
print("Test raw ROC-AUC:", roc_auc_score(y_test, test_proba_raw))


Train raw PR-AUC: 1.0
Train raw ROC-AUC: 1.0
Test raw PR-AUC: 0.125
Test raw ROC-AUC: 0.86


---
## CELL 6.7: Calibrate with Platt Scaling (LogisticRegression)


In [20]:
X_cal = train_proba_raw.reshape(-1, 1)
platt = LogisticRegression(solver="lbfgs", max_iter=2000)
platt.fit(X_cal, y_train)

train_proba_cal = platt.predict_proba(X_cal)[:, 1]
test_proba_cal = platt.predict_proba(test_proba_raw.reshape(-1,1))[:,1]

metrics = {
    "train_raw": {
        "pr_auc": float(average_precision_score(y_train, train_proba_raw)),
        "roc_auc": float(roc_auc_score(y_train, train_proba_raw)),
        "brier": float(brier_score_loss(y_train, train_proba_raw)),
    },
    "train_cal": {
        "pr_auc": float(average_precision_score(y_train, train_proba_cal)),
        "roc_auc": float(roc_auc_score(y_train, train_proba_cal)),
        "brier": float(brier_score_loss(y_train, train_proba_cal)),
    },
    "test_raw": {
        "pr_auc": float(average_precision_score(y_test, test_proba_raw)),
        "roc_auc": float(roc_auc_score(y_test, test_proba_raw)),
        "brier": float(brier_score_loss(y_test, test_proba_raw)),
    },
    "test_cal": {
        "pr_auc": float(average_precision_score(y_test, test_proba_cal)),
        "roc_auc": float(roc_auc_score(y_test, test_proba_cal)),
        "brier": float(brier_score_loss(y_test, test_proba_cal)),
    }
}

print(json.dumps(metrics, indent=2))

with open("results/reports/section6_calibration_metrics.json", "w") as f:
    json.dump({
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "warning": "Only 3 positives total; treat metrics as exploratory.",
        "metrics": metrics
    }, f, indent=2)


{
  "train_raw": {
    "pr_auc": 1.0,
    "roc_auc": 1.0,
    "brier": 0.002843315014615655
  },
  "train_cal": {
    "pr_auc": 1.0,
    "roc_auc": 1.0,
    "brier": 0.0091311583028613
  },
  "test_raw": {
    "pr_auc": 0.125,
    "roc_auc": 0.86,
    "brier": 0.03473365306854248
  },
  "test_cal": {
    "pr_auc": 0.125,
    "roc_auc": 0.86,
    "brier": 0.019356668246271063
  }
}


---
## CELL 6.8: Define Cross-Reactivity Probability

Heuristic definition:
- High cross-reactivity ≈ high IgM/IgG OD values **and** PCR negative.
- We approximate `p_cross_reactive(x)` as:
  - take all **training negatives**
  - fit a simple LogisticRegression using IgM/IgG OD values to predict “high_OD” label
  - high_OD label = 1 if IgM OD or IgG OD above chosen percentile (e.g. 90th) among negatives

Then at inference:
- `p_cross_reactive(x)` = calibrated probability of being in this “high OD” group while PCR negative.

Note: This is a rough proxy but gives you a **separate score for cross-reactive patterns**.


In [22]:
# -------------------------------------------------------------------
# Robust detection of IgM / IgG OD columns
# -------------------------------------------------------------------
neg_mask_train = (y_train == 0)
X_train_neg = X_train.loc[neg_mask_train].copy()

print("All columns in X_train_neg:")
print(list(X_train_neg.columns))

# Try to auto-detect IgM and IgG OD columns by partial match
igm_candidates = [c for c in X_train_neg.columns 
                  if "IgM" in c and "OD" in c]
igg_candidates = [c for c in X_train_neg.columns 
                  if "IgG" in c and "OD" in c]

print("\nDetected IgM OD candidates:", igm_candidates)
print("Detected IgG OD candidates:", igg_candidates)

if not igm_candidates or not igg_candidates:
    raise KeyError(
        f"Could not automatically find IgM/IgG OD columns. "
        f"Found IgM candidates={igm_candidates}, IgG candidates={igg_candidates}"
    )

# If multiple, pick the first; you can override manually if needed
igm_col = igm_candidates[0]
igg_col = igg_candidates[0]

print(f"\nUsing IgM column: {igm_col}")
print(f"Using IgG column: {igg_col}")

# -------------------------------------------------------------------
# Define high-OD based on 90th percentile among training negatives
# -------------------------------------------------------------------
igm_thr = X_train_neg[igm_col].quantile(0.90)
igg_thr = X_train_neg[igg_col].quantile(0.90)

X_od = X_train_neg[[igm_col, igg_col]].values
y_od = ((X_train_neg[igm_col] >= igm_thr) | 
        (X_train_neg[igg_col] >= igg_thr)).astype(int).values

from sklearn.linear_model import LogisticRegression

cross_model = LogisticRegression(solver="lbfgs", max_iter=2000)
cross_model.fit(X_od, y_od)

print("\nHigh-OD prevalence among negatives:", y_od.mean())
print("IgM high threshold:", igm_thr)
print("IgG high threshold:", igg_thr)

All columns in X_train_neg:
['Age', 'Gender', 'State', 'Country', 'Settlement_Type', 'Recent_Travel_30D', 'Rodent_Contact_6M', 'Food_Open_Storage', 'Rodents_Droppings_home', 'Participated_in_Survey', 'Hopital_Visits_30D', 'Contact_Confirmed_Case', 'LASV_Mortality', 'Community_lassa_awareness', 'clinical_symptoms.Fever', 'clinical_symptoms.Headache ', 'clinical_symptoms.Muscle Pains', 'clinical_symptoms.Weakness', 'clinical_symptoms.Sore Throat', 'clinical_symptoms.Vomiting', 'clinical_symptoms.Nausea', 'clinical_symptoms.Abdominal Pain', 'clinical_symptoms.Diarrhea', 'clinical_symptoms.Facial Swelling ', 'clinical_symptoms.Bleeding ( Nose, Gum, Stool)', 'clinical_symptoms.Difficult Breathing ', 'clinical_symptoms.Chest Pain ', 'clinical_symptoms.Hearing Loss', 'clinical_symptoms.Skin Rash ', 'clinical_symptoms.Pre_Existing_Conditions', 'clinical_symptoms.Vaccinated_in_Past_Years', 'clinical_symptoms.Current_Medications', 'treatment_outcomes.Hospitalized', 'treatment_outcomes.Hospital_D

---
## CELL 6.9: Combine Outputs for New Patients

Given a new patient **x**:
1. Compute `Xt = preprocess.transform(x)`
2. `raw_proba = mean_m.predict_proba(Xt)[:,1]` over ensemble
3. `p_pcr_pos = platt.predict_proba(raw_proba.reshape(-1,1))[:,1]`
4. `p_cross_reactive = cross_model.predict_proba([[IgM_OD, IgG_OD]])[:,1]`

We also compute **a simple suggested label**:
- If `p_pcr_pos` >= threshold (e.g. balanced F1), label = *PCR Positive (low cross-reactivity if p_cross_reactive is low)*
- Else label = *PCR Negative (check cross-reactivity — high if p_cross_reactive is high)*

We will search a reasonable threshold on TRAIN set (calibrated proba).


In [23]:
def threshold_sweep(y_true, proba, n=300):
    thresholds = np.linspace(0.001, 0.5, n)
    rows = []
    for t in thresholds:
        y_hat = (proba >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "f1": float(f1_score(y_true, y_hat, zero_division=0)),
            "precision": float(precision_score(y_true, y_hat, zero_division=0)),
            "recall": float(recall_score(y_true, y_hat, zero_division=0)),
            "predicted_positives": int(y_hat.sum())
        })
    df = pd.DataFrame(rows)
    best_f1 = df.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    return df, best_f1

sweep_df, best_th = threshold_sweep(y_train, train_proba_cal)
print("Best train threshold (calibrated):")
print(best_th)
sweep_df.to_csv("results/reports/section6_threshold_sweep_train_calibrated.csv", index=False)

best_threshold = float(best_th["threshold"])


Best train threshold (calibrated):
threshold              0.024365
f1                     1.000000
precision              1.000000
recall                 1.000000
predicted_positives    2.000000
Name: 14, dtype: float64


---
## CELL 6.10: Evaluate on TEST with Combined Logic


In [24]:
# PCR probability (calibrated)
y_test_hat = (test_proba_cal >= best_threshold).astype(int)

print("Test confusion matrix (PCR labels):")
print(confusion_matrix(y_test, y_test_hat))
print("Test precision, recall, f1 (pos=1):")
print("precision=", precision_score(y_test, y_test_hat, zero_division=0))
print("recall   =", recall_score(y_test, y_test_hat, zero_division=0))
print("f1       =", f1_score(y_test, y_test_hat, zero_division=0))


Test confusion matrix (PCR labels):
[[49  1]
 [ 1  0]]
Test precision, recall, f1 (pos=1):
precision= 0.0
recall   = 0.0
f1       = 0.0


---
## CELL 6.11: Save Final Artifacts for Deployment

We save:
- `models/section6_preprocess_drop.joblib`
- `models/section6_xgb_ensemble.joblib` (list of models)
- `models/section6_platt_calibrator.joblib`
- `models/section6_cross_reactivity_model.joblib`
- `results/reports/section6_inference_config.json` with thresholds + columns


In [ ]:
# Save preprocess
joblib.dump(preprocess, "models/section6_preprocess_drop.joblib")

# Save ensemble as list
joblib.dump(ensemble_models, "models/section6_xgb_ensemble.joblib")

# Save calibrator
joblib.dump(platt, "models/section6_platt_calibrator.joblib")

# Save cross-reactivity model + thresholds
cross_artifacts = {
    "model": cross_model,
    "igm_col": igm_col,
    "igg_col": igg_col,
    "igm_thr": float(igm_thr),
    "igg_thr": float(igg_thr)
}
joblib.dump(cross_artifacts, "models/section6_cross_reactivity_model.joblib")

config = {
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_threshold_calibrated": best_threshold,
    "notes": {
        "warning": "Only 3 positives; predictive power + stability limited.",
        "intended_use": "Research / development of PCR + cross-reactivity scorer",
        "pcr_probability_name": "p_pcr_pos",
        "cross_reactivity_probability_name": "p_cross_reactive"
    },
    "columns": {
        "numeric": num_cols,
        "categorical": cat_cols
    }
}

with open("results/reports/section6_inference_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✓ Saved artifacts for Section 6")
print("  models/section6_preprocess_drop.joblib")
print("  models/section6_xgb_ensemble.joblib")
print("  models/section6_platt_calibrator.joblib")
print("  models/section6_cross_reactivity_model.joblib")
print("  results/reports/section6_inference_config.json")


---
## CELL 6.12: Inference Helper Function (for Future Deployment)

This is the exact logic you will reuse in the web app.


In [ ]:
def load_section6_artifacts():
    pre = joblib.load("models/section6_preprocess_drop.joblib")
    ensemble = joblib.load("models/section6_xgb_ensemble.joblib")
    pl = joblib.load("models/section6_platt_calibrator.joblib")
    cross = joblib.load("models/section6_cross_reactivity_model.joblib")
    with open("results/reports/section6_inference_config.json", "r") as f:
        cfg = json.load(f)
    return pre, ensemble, pl, cross, cfg

def section6_predict(df_input: pd.DataFrame):
    """Given a DataFrame with same columns as training X,
    return p_pcr_pos, p_cross_reactive, and suggested labels."""
    pre, ensemble, pl, cross, cfg = load_section6_artifacts()

    # ensure columns order & presence
    missing_cols = [c for c in X.columns if c not in df_input.columns]
    if missing_cols:
        raise ValueError(f"Input is missing columns: {missing_cols}")
    df_input = df_input[X.columns]  # align

    Xt = pre.transform(df_input)
    raw_proba = ensemble_predict_proba(ensemble, Xt)
    p_pcr_pos = pl.predict_proba(raw_proba.reshape(-1,1))[:,1]

    igm_col = cross["igm_col"]
    igg_col = cross["igg_col"]
    if igm_col not in df_input.columns or igg_col not in df_input.columns:
        raise KeyError("IgM/IgG OD columns missing from input.")

    X_od_new = df_input[[igm_col, igg_col]].values
    p_high_od = cross["model"].predict_proba(X_od_new)[:,1]
    p_cross_reactive = p_high_od

    thr = cfg["best_threshold_calibrated"]
    y_hat = (p_pcr_pos >= thr).astype(int)

    labels = []
    for pp, pc, lab in zip(p_pcr_pos, p_cross_reactive, y_hat):
        if lab == 1:
            if pc < 0.5:
                labels.append("Likely PCR Positive (low cross-reactivity)")
            else:
                labels.append("PCR Positive (but high OD; consider cross-reactivity)")
        else:
            if pc >= 0.5:
                labels.append("PCR Negative (high cross-reactivity / high OD background)")
            else:
                labels.append("PCR Negative (low cross-reactivity)")

    out = df_input.copy()
    out["p_pcr_pos"] = p_pcr_pos
    out["p_cross_reactive"] = p_cross_reactive
    out["prediction_label"] = labels
    return out

print("Helper function section6_predict(df_input) is defined and ready to use.")
